In [2]:
%pip install git+https://github.com/huggingface/trl.git

  Cloning https://github.com/huggingface/trl.git to /tmp/pip-req-build-ehe1ugj1
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/trl.git /tmp/pip-req-build-ehe1ugj1
  Resolved https://github.com/huggingface/trl.git to commit 6f99f42f724123409422f2fad42bf56fa91f366f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.


In [49]:
%pip uninstall -y trl

Found existing installation: trl 0.14.0.dev0
Uninstalling trl-0.14.0.dev0:
  Successfully uninstalled trl-0.14.0.dev0
Note: you may need to restart the kernel to use updated packages.


In [4]:
%env CUDA_HOME=/opt/conda/lib/python3.11/site-packages/torch/cuda
%pip install trl accelerate vllm transformer_engine[pytorch]

env: CUDA_HOME=/opt/conda/lib/python3.11/site-packages/torch/cuda
  Using cached transformer_engine-1.13.0-py3-none-any.whl.metadata (16 kB)
  Using cached transformer_engine_cu12-1.13.0-py3-none-manylinux_2_28_x86_64.whl.metadata (16 kB)
  Using cached transformer_engine_torch-1.13.0.tar.gz (121 kB)
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.2/125.2 MB 16.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 460.0/460.0 kB 4.1 MB/s eta 0:00:00:00:01
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> [25 lines of output]
      Could not determine CUDA Toolkit version
      running bdist_wheel
      /opt/conda/lib/python3.11/site-packages/torch/utils/cpp_extension.py:497: UserWarning: Attempted to use ninja as the BuildExtension backend but we could not find ninja.. Falling back to using the slow distutils backend.
        warnings.warn(msg.format('we c

In [2]:
import datasets
updated_dataset = datasets.load_dataset("w601sxs/processed_simpleCoT_b1ade", split="train")

In [3]:
updated_dataset

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 2214941
})

In [4]:
# updated_dataset = dataset.map(lambda example: {
#     'prompt': f"context: <You are a useful AI assistant. Use the provided context to provide a rationale, and then answer the question that follows.>\nquestion: <{example['source']}>\n",
#     'completion': f"answer: <{example['rationale']}\n{example['target']}>" 
# })

In [5]:
# updated_dataset = updated_dataset.remove_columns(['source', 'target', 'rationale', 'task', 'type'])

In [6]:
# %pip install -U "huggingface_hub[cli]"

In [7]:
# updated_dataset.push_to_hub("w601sxs/processed_simpleCoT_b1ade")

In [8]:
# %pip install peft

In [15]:
%%writefile train_grpo.py
from datasets import load_dataset
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer
import pandas as pd
from sentence_transformers import SentenceTransformer
emodel = SentenceTransformer("w601sxs/b1ade-embed", device='cuda')
import re
import time
import re
import torch

def format_reward_func(completions):
    """Reward function that checks if the completion has a specific format."""
    pattern = r'^answer: <.*>$'
    matches = [re.match(pattern, content, re.DOTALL) for content in completions]
    return [1.0 if match else 0.0 for match in matches]

def reward_func(prompts, completions):
    """Reward function that gives higher scores to longer completions."""
    format_reward = torch.Tensor(format_reward_func(completions))
    
    
    return answer_similarity(prompts, completions)*format_reward

# Load the dataset
updated_dataset = load_dataset("w601sxs/processed_simpleCoT_b1ade", split="train")

def extract_content(text):
    match = re.match(r'^answer: <(.*)>$', text, re.DOTALL)
    return match.group(1) if match else text

df = updated_dataset.to_pandas()

def answer_similarity(prompts, completions):
    # Extract ground truth completions
    ground_truth_completions = []
    for prompt in prompts:
        row = df.loc[df['prompt'] == prompt]
        if not row.empty:
            gtcompletion = row['completion'].iloc[0]
            ground_truth_completions.append(extract_content(gtcompletion))
        else:
            ground_truth_completions.append(None)
    
    
    # Process input completions
    processed_completions = [extract_content(comp) for comp in completions]
    
    # Filter out None values
    valid_mask = [gt is not None for gt in ground_truth_completions]
    valid_ground_truth = [gt for gt in ground_truth_completions if gt is not None]
    valid_completions = [comp for comp, keep in zip(processed_completions, valid_mask) if keep]

    # Calculate embeddings and similarities
    completion_embeddings = emodel.encode(valid_completions)
    ground_truth_embeddings = emodel.encode(valid_ground_truth)
    similarities = emodel.similarity(completion_embeddings, ground_truth_embeddings).diagonal()
    
    return similarities


training_args = GRPOConfig(
    output_dir="b1ade-1b-GRPO",
    learning_rate=1e-5,
    logging_steps=1,
    max_completion_length=256,
    bf16=True,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=10,
    use_vllm=True,
    vllm_device="auto")

trainer = GRPOTrainer(
    model="w601sxs/b1ade-1b-bf16",
    reward_funcs=reward_func,
    args=training_args,
    train_dataset=updated_dataset,
    peft_config=LoraConfig(task_type="CAUSAL_LM", r=4),
)

trainer.train()

Overwriting train_grpo.py


In [16]:
import torch
torch.cuda.device_count()

4

In [18]:
!accelerate launch --num_machines 1 --num_processes 3 --downcast_bf16 --multi_gpu --mixed_precision 'fp8' train_grpo.py

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Traceback (most recent call last):
  File "/opt/conda/bin/accelerate", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/accelerate/commands/accelerate_cli.py", line 48, in main
    args.func(args)
  File "/opt/conda/lib/python3.11/site-packages/accelerate/commands/launch.py", line 1163, in launch_command
    multi_gpu_launcher(args)
  File "/opt/conda/lib/python3.11/site-packages/accelerate/commands/launch.py", line 770, in multi_gpu_launcher
    current_env = prepare_multi_gpu_env(args)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/accelerate/utils/launch.py", line 249, in prepare_multi_gpu_env
    raise RuntimeError(
RuntimeError: FP8 is 

In [27]:
import torch

torch._dynamo.list_backends()

['cudagraphs', 'inductor', 'onnxrt', 'openxla', 'tvm']